In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

# Load the dataset
data = load_breast_cancer()
X = data.data
y = data.target

# Split into an 80/20 training/test set
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
from sklearn.preprocessing import StandardScaler
# Standardize features using training data only
# Have to do this otherwise the sgd's don't converge
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Display dataset characteristics
print(f"Feature shape: {X.shape}")
print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Target classes: {data.target_names}")

Feature shape: (569, 30)
Training set shape: (455, 30)
Test set shape: (114, 30)
Target classes: ['malignant' 'benign']


# Classification Part A

# SK Learn's Logistic Regression

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Initialize and train the model
# We increase max_iter to ensure convergence on this dataset
model = LogisticRegression(max_iter=10000, random_state=42)
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Calculate and print the test accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 0.9825


# My Logistic Regression

### Look at our data shape

In [3]:
print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Target classes: {data.target_names}")

Training set shape: (455, 30)
Test set shape: (114, 30)
Target classes: ['malignant' 'benign']


### Prep the data

In [4]:
import torch

torch.manual_seed(420)



# Convert dataset into PyTorch tensors with readable names
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

### Define the logistic regression model

In [5]:
# Initialize weights w ∈ R^30 and bias b in R
# requires_grad=True so torch knows we diff w.r.t. this
w = torch.zeros(30, requires_grad=True)
b = torch.zeros((), requires_grad=True)

# Note that w^T x is a the linear regressor but we feed it into a function
# to trunc the values
# Model: f(x) = (w^T x + b)
def f(X):
    return torch.sigmoid(X @ w + b)

### Define loss function and optimizer

In [6]:
# Binary cross-entropy loss (values are always >= 0)
def cross_entropy(y, p):
    p = p.clamp(1e-6, 1 - 1e-6)  # Avoid log(0)
    return -(y * torch.log(p) + (1-y) * torch.log(1-p)).mean()

# SGD: theta_new = theta_old - ('lr')('gradL'), where theta = (w, b)
optimizer = torch.optim.SGD([w, b], lr=1e-3)

### Train the model

In [7]:
# Sample a batch, predict, calculate the loss on this batch,
# calculate the gradients which point towards steepest descent
#  take a small step and
# repeat.
for epoch in range(500):
    # Sample 32 X,y pairs.
    for idx in torch.randperm(len(X_train_tensor)).split(32):
        # Forward pass: predict and calc loss
        predictions = f(X_train_tensor[idx])
        loss = cross_entropy(y_train_tensor[idx], predictions)

        # Back pass, clear out old grads
        # compute dL/dW and dl/dB
        optimizer.zero_grad()
        loss.backward()

        # step in the negative gradient
        # the negative aspect is baked
        # into optimizer by defualt
        optimizer.step()

### Compare Mine vs SKLearns

In [8]:
# Evaluate on test set without tracking gradients
with torch.no_grad():
    predictions = (f(X_test_tensor) >= 0.5).float()          # Convert to binary labels
    accuracy = (predictions == y_test_tensor).float().mean() # Calculate accuracy

print("PyTorch accuracy:", accuracy.item())
print("sklearn accuracy:", model.score(X_test, y_test))

PyTorch accuracy: 0.9649122953414917
sklearn accuracy: 0.9824561403508771


# Classification Part (B)

## My Probit func

In [9]:
# New parameters, separate from logistic regression
w_probit = torch.zeros(30, requires_grad=True)
b_probit = torch.zeros((), requires_grad=True)

# Std normal N(0,1)
normal = torch.distributions.Normal(0, 1)

# Model: f(x) = (w^T x + b)
def f_probit(X):
    return normal.cdf(X @ w_probit + b_probit)

In [10]:
# Optimizer for the probit version
optimizer_probit = torch.optim.SGD([w_probit, b_probit], lr=1e-3)

In [11]:
# Train for 300 epochs
for epoch in range(300):
    for idx in torch.randperm(len(X_train_tensor)).split(32):
        predictions = f_probit(X_train_tensor[idx])
        loss = cross_entropy(y_train_tensor[idx], predictions)

        optimizer_probit.zero_grad()
        loss.backward()
        optimizer_probit.step()

## Evaluate and Compare

In [12]:
# Evaluate on test set
with torch.no_grad():
    predictions = (f_probit(X_test_tensor) >= 0.5).float()
    accuracy_probit = (predictions == y_test_tensor).float().mean()

print("My probit accuracy:", accuracy_probit.item())
print("SKlearn baseline:", model.score(X_test, y_test))

My probit accuracy: 0.9736841917037964
SKlearn baseline: 0.9824561403508771


# Regression Part (A)

In [13]:
from google.colab import drive
import pandas as pd
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/Datasets/Homework1/ave.csv')
df = df.drop(columns=['sd_ale'])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
from sklearn.model_selection import train_test_split

# Separate features and target (ALE)
X = df.drop(columns=['ale']).to_numpy()
y = df['ale'].to_numpy()

# Split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

from sklearn.preprocessing import StandardScaler
# Standardize features using training data only
# Have to do this otherwise the sgd's don't converge
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## SKLeaern Regression

In [15]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# Train linear regression
model = LinearRegression()
model.fit(X_train, y_train)

# Evaluate using MAE
y_pred = model.predict(X_test)
baseline_mae = mean_absolute_error(y_test, y_pred)

print("sklearn MAE:", baseline_mae)

sklearn MAE: 0.1953934164192885


## My regressor

In [16]:
# Convert data into PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

In [17]:
# Initialize weights w in R^4 and bias b in R
w = torch.zeros(4, requires_grad=True)
b = torch.zeros((), requires_grad=True)

# Model: f(x) = w^Tx + b
def f(X):
    return X @ w + b

MSE Loss Func


In [18]:
# Mean squared error: L = (y - f(x))^2
def mse(y, predictions):
    return ((y - predictions) ** 2).mean()

# SGD optimizer
optimizer = torch.optim.SGD([w, b], lr=0.001)

## Train

In [19]:
for epoch in range(3000):
    # Shuffle observations into batches of 32
    for idx in torch.randperm(len(X_train_tensor)).split(32):
        predictions = f(X_train_tensor[idx])
        loss = mse(y_train_tensor[idx], predictions)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

## Compare vs SK

In [20]:
# Evaluate on test set
with torch.no_grad():
    predictions = f(X_test_tensor)
    mae = (predictions - y_test_tensor).abs().mean()

print("PyTorch MAE:", mae.item())
print("sklearn MAE:", baseline_mae)

PyTorch MAE: 0.19534224271774292
sklearn MAE: 0.1953934164192885


## Regression Part (B)

In [21]:
# Initialize separate weights and bias
w_t = torch.zeros(X_train_tensor.shape[1], requires_grad=True)
b_t = torch.zeros((), requires_grad=True)

# Model: f(x) = wፔx + b
def f_t(X):
    return X @ w_t + b_t

## Student T Loss Func

In [22]:
# Student's t-distribution with 5 degrees of freedom
t_dist = torch.distributions.StudentT(df=5)

# Negative log-likelihood: L = -log p(y - f(x), 5)
def t_loss(y, predictions):
    residuals = y - predictions
    return -t_dist.log_prob(residuals).mean()

# SGD optimizer
optimizer_t = torch.optim.SGD([w_t, b_t], lr=0.01)

In [23]:
for epoch in range(2000):
    for idx in torch.randperm(len(X_train_tensor)).split(32):
        predictions = f_t(X_train_tensor[idx])
        loss = t_loss(y_train_tensor[idx], predictions)

        optimizer_t.zero_grad()
        loss.backward()
        optimizer_t.step()

In [24]:
# Evaluate using mean absolute error
with torch.no_grad():
    predictions_t = f_t(X_test_tensor)
    mae_t = (predictions_t - y_test_tensor).abs().mean()

print("sklearn MAE:", baseline_mae)
print("PyTorch MSE MAE:", mae.item())
print("PyTorch Student's t MAE:", mae_t.item())

sklearn MAE: 0.1953934164192885
PyTorch MSE MAE: 0.19534224271774292
PyTorch Student's t MAE: 0.19401730597019196
